# ExtraTrees — Optuna hyperparameter study on `fe_v0_native` (CPU, ~12 h)

80-trial Optuna study for `ExtraTreesClassifier` on the **native base features** — the
19 raw columns with **no feature engineering** (the `fe_v0` baseline).

**Why ordinal encoding?** `sklearn`'s `ExtraTreesClassifier` has no native categorical
path. Each `category`-dtype column is converted to its `cat.codes` integer before
training. Leakage-free: `cat.codes` depends only on the dtype categories (defined by
`prepare_data` over the full dataset), not on any per-fold statistic.

**Why CPU?** ExtraTrees is embarrassingly parallel across trees (`n_jobs=-1`) and gets no
benefit from a GPU. CPU quota is effectively unlimited compared to GPU hours.

**Settings (right sidebar):** Accelerator → **None / CPU**; Internet → **On**; Add Input →
**playground-series-s6e3**.

> ⏱️ Each trial runs 3-fold inner CV with `n_jobs=-1`. ExtraTrees fits faster than RF
> (randomized splits, no exhaustive best-split search). **SMOKE-TEST FIRST:** set
> `n_trials=4` to measure per-trial time before the full 80-trial Save & Run All.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell.
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...


CWD: /kaggle/working/Predict-Customer-Churn


Updating files: 100% (354/354), done.


In [3]:
# Kaggle's image ships sklearn; only optuna may be missing.
!pip install -q optuna

import sklearn, optuna
print("sklearn:", sklearn.__version__)
print("optuna: ", optuna.__version__)
print("CPU cores:", os.cpu_count())

# Quick sanity fit to confirm ExtraTreesClassifier works before the full study.
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier

_Xs = np.random.rand(2000, 6)
_ys = (np.random.rand(2000) > 0.5).astype(int)
ExtraTreesClassifier(n_estimators=5, max_depth=5, random_state=42).fit(_Xs, _ys)
print("ExtraTreesClassifier CPU OK")

sklearn: 1.6.1
optuna:  4.8.0
CPU cores: 4
ExtraTreesClassifier CPU OK


In [4]:
# data/processed/*.parquet are tracked in the repo, so prepare_data loads them
# directly from the clone (data_hash then matches local runs). Copy the raw CSVs
# as well so prepare_data can rebuild from source if the cache is ever absent.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native')
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Loaded from cache (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


In [5]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import DATA_DIR, RUNS_DIR, RUNS_CSV
from src.cv import run_cv_experiment, save_experiment

### Feature setup — ordinal-encode the native base (`fe_v0_native`)

In [6]:
DATA_VERSION = 'fe_v0_native'

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train_base = train_df[encoded_features]
y_train      = train_df['Churn']
X_test_base  = test_df[encoded_features]

# --- Ordinal-encode categorical columns for ExtraTreesClassifier ---
# sklearn's ExtraTrees rejects category-dtype columns. cat.codes converts each level
# to its integer position (0, 1, 2, ...); unknown test levels become -1. Alignment via
# set_categories guarantees train and test use identical codes for shared levels. This
# is leakage-free: codes depend only on the dtype categories (from prepare_data over the
# full dataset), not on any per-fold statistic.
cat_cols = [c for c in encoded_features
            if isinstance(X_train_base[c].dtype, pd.CategoricalDtype)]
print(f'Ordinal-encoding {len(cat_cols)} categorical columns: {cat_cols}')

X_train = X_train_base.copy()
X_test  = X_test_base.copy()
for col in cat_cols:
    X_test[col]  = X_test[col].cat.set_categories(X_train[col].cat.categories)
    X_train[col] = X_train[col].cat.codes
    X_test[col]  = X_test[col].cat.codes

print(f'After encoding: X_train {X_train.shape}  X_test {X_test.shape}  features: {len(encoded_features)}')

Ordinal-encoding 15 categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
After encoding: X_train (594194, 19)  X_test (254655, 19)  features: 19


### Optuna study — ExtraTrees on `fe_v0_native`

In [7]:
import optuna
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

et_inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


def et_objective(trial):
    # max_features: two named shortcuts plus a searched float fraction.
    mf_choice = trial.suggest_categorical('max_features_type', ['sqrt', 'log2', 'float'])
    if mf_choice == 'float':
        max_features = trial.suggest_float('max_features_float', 0.05, 0.7)
    else:
        max_features = mf_choice

    # ExtraTrees defaults to bootstrap=False (uses the whole sample with randomized
    # splits). max_samples is only valid when bootstrap=True, so it is suggested
    # conditionally inside that branch.
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 25),
        'max_features':      max_features,
        'bootstrap':         bootstrap,
        'n_jobs':            -1,      # parallelize tree building across all vCPUs
        'random_state':      42,
    }
    if bootstrap:
        params['max_samples'] = trial.suggest_float('max_samples', 0.4, 1.0)

    scores = cross_val_score(
        ExtraTreesClassifier(**params), X_train, y_train,
        cv=et_inner_cv, scoring='roc_auc',
        n_jobs=1,  # one trial at a time; ET already uses all cores via n_jobs=-1
    )
    return scores.mean()


et_study = optuna.create_study(
    study_name='et-fe_v0_native',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Warm-start from ExtraTrees' canonical defaults: sqrt features, no bootstrap.
et_study.enqueue_trial({
    'n_estimators':      200,
    'max_depth':         15,
    'min_samples_split': 2,
    'min_samples_leaf':  1,
    'max_features_type': 'sqrt',
    'bootstrap':         False,
})

# SMOKE-TEST: set n_trials=4 first to confirm per-trial timing, then restore 80.
et_study.optimize(et_objective, n_trials=80, show_progress_bar=True)

print(f'Best inner-CV ROC AUC: {et_study.best_value:.6f}  (trial {et_study.best_trial.number})')
for k, v in et_study.best_params.items():
    print(f'  {k:25s} {v}')

  0%|          | 0/80 [00:00<?, ?it/s]

Best inner-CV ROC AUC: 0.912594  (trial 67)
  max_features_type         float
  max_features_float        0.660162480765257
  bootstrap                 True
  n_estimators              473
  max_depth                 19
  min_samples_split         2
  min_samples_leaf          24
  max_samples               0.4247212543111798


### Run configuration

Best Optuna params are reassembled and passed to `run_cv_experiment`, which fits the
5-fold outer CV and prints OOF accuracy + ROC-AUC. `data_version` ties the run back to
the underlying feature parquet.

In [8]:
# Reconstruct max_features from the split Optuna params.
_mf_type  = et_study.best_params.get('max_features_type')
_mf_float = et_study.best_params.get('max_features_float')
_best_max_features = _mf_float if _mf_type == 'float' else _mf_type

_et_params = {k: v for k, v in et_study.best_params.items()
              if k not in ('max_features_type', 'max_features_float')}
_et_params['max_features'] = _best_max_features
_et_params['n_jobs']       = -1
_et_params['random_state'] = 42
# 'bootstrap' and (if present) 'max_samples' pass straight through to ExtraTrees.

print('Best ExtraTrees params:')
for k, v in _et_params.items():
    print(f'  {k:22s} {v}')

run_config = {
    'model_factory': lambda params: ExtraTreesClassifier(**params),
    'params':        _et_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'et-optuna-fe_v0_native',
    'notes':         'ExtraTreesClassifier on fe_v0_native (19 raw native categorical/numeric features, no feature engineering). Category-dtype columns ordinal-encoded via cat.codes before training (train/test aligned with set_categories). bootstrap searched as a categorical; max_samples tuned only when bootstrap=True. Best params from an 80-trial Optuna study (3-fold inner CV, ROC-AUC, TPE); trial 0 seeded from ExtraTrees canonical defaults. Notebook: kaggle/predict-customer-churn-et-cpu-fe_v0.ipynb.',
    'parent_run_id': '',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

Best ExtraTrees params:
  bootstrap              True
  n_estimators           473
  max_depth              19
  min_samples_split      2
  min_samples_leaf       24
  max_samples            0.4247212543111798
  max_features           0.660162480765257
  n_jobs                 -1
  random_state           42


In [9]:
# Step 1 — Run the experiment (fits 5 folds, prints OOF accuracy + ROC-AUC).
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-235928-72aa26
Tag:    et-optuna-fe_v0_native

Fold 0: accuracy=0.8577  roc_auc=0.9123  (fit 92.8s)
Fold 1: accuracy=0.8581  roc_auc=0.9137  (fit 98.9s)
Fold 2: accuracy=0.8577  roc_auc=0.9127  (fit 99.0s)
Fold 3: accuracy=0.8590  roc_auc=0.9140  (fit 91.1s)
Fold 4: accuracy=0.8574  roc_auc=0.9112  (fit 95.1s)

OOF accuracy: 0.8580
OOF ROC-AUC:  0.9128
Folds:        0.8580 ± 0.0005

Run complete. Call save_experiment(result) to log this run permanently.


In [10]:
# Step 2 — Save the run (review the OOF ROC-AUC above first).
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-235928-72aa26


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds) churn probability for the full test
set. The competition metric is ROC-AUC, so submit the probability directly.

In [11]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

       id     Churn
0  594194  0.057863
1  594195  0.000150
2  594196  0.115407
3  594197  0.001855
4  594198  0.485233
wrote /kaggle/working/submission.csv (254655, 2)


### Bundle run artifacts + source notebook into one zip

Zips the run directory, `runs.csv`, and the source `.ipynb` into a single archive
on the Output tab. To fold the run back into the local repo, follow
**§7-8 of `docs/kaggle_gpu_workflow.md`**.

In [12]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')
# 3) source notebook committed in the cloned repo
src_nb = Path(REPO_ROOT) / 'kaggle' / 'predict-customer-churn-et-cpu-fe_v0.ipynb'
if src_nb.exists():
    shutil.copy(src_nb, BUNDLE / src_nb.name)
    print('bundled notebook:', src_nb.name)
else:
    print('source notebook not found in clone (push it to master first for future runs)')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

source notebook not found in clone (push it to master first for future runs)
wrote /kaggle/working/20260612-235928-72aa26_bundle.zip
